# 📊 FinTrack — Exploratory Data Analysis (EDA)
**Dicoding Capstone 2026 | Data Science Path**

Notebook ini berisi analisis eksploratif dataset transaksi keuangan untuk memahami pola pengeluaran pengguna.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

plt.style.use('dark_background')
sns.set_palette('Blues_r')
%matplotlib inline

df = pd.read_csv('../dataset/transactions_clean.csv', parse_dates=['date'])
print(f'Shape: {df.shape}')
print(f'Kolom: {list(df.columns)}')
display(df.head(10))

## 1. Statistik Deskriptif

In [ ]:
display(df.describe())
print(f"\nMissing values:\n{df.isnull().sum()}")

## 2. Distribusi Pengeluaran per Kategori

In [ ]:
expense = df[df['type']=='expense']
cat_total = expense.groupby('category')['amount'].sum().sort_values(ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
cat_total.plot(kind='bar', ax=ax1, color='steelblue')
ax1.set_title('Total Pengeluaran per Kategori')
ax1.set_ylabel('Total (Rp)')
ax1.tick_params(axis='x', rotation=45)

category_counts = expense['category'].value_counts()
ax2.pie(category_counts, labels=category_counts.index, autopct='%1.1f%%', startangle=90)
ax2.set_title('Proporsi Frekuensi Transaksi')
plt.tight_layout()
plt.savefig('../reports/01_category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Pola Waktu Transaksi Impulsif

In [ ]:
imp = df[df['is_impulsive']==True]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
hour_counts = imp['hour'].value_counts().sort_index()
hour_counts.plot(kind='bar', ax=ax1, color='salmon')
ax1.set_title('Transaksi Impulsif per Jam')
ax1.set_xlabel('Jam')
ax1.set_ylabel('Jumlah Transaksi')
ax1.axvspan(21, 23, alpha=0.2, color='red', label='Jam rawan (21-23)')

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = imp['weekday'].value_counts().reindex(day_order)
day_counts.plot(kind='bar', ax=ax2, color='orange')
ax2.set_title('Transaksi Impulsif per Hari')
ax2.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../reports/02_impulse_patterns.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total impulsif: {len(imp)} ({len(imp)/len(df)*100:.1f}%)')

## 4. Tren Bulanan

In [ ]:
monthly = df.groupby(['month','type'])['amount'].sum().unstack().fillna(0)
monthly.plot(kind='bar', figsize=(12, 5), width=0.8)
plt.title('Pemasukan vs Pengeluaran per Bulan')
plt.ylabel('Total (Rp)')
plt.xticks(rotation=45)
plt.legend(['Pengeluaran','Pemasukan'])
plt.tight_layout()
plt.savefig('../reports/03_monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Korelasi Antar Variabel

In [ ]:
df_num = df[['amount','hour','is_impulsive']].copy()
corr = df_num.corr()
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Korelasi Antar Variabel')
plt.tight_layout()
plt.savefig('../reports/04_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKonklusi EDA:')
print('- Transaksi impulsif dominan di jam 21-23')
print('- Hari Jumat & Sabtu memiliki transaksi impulsif tertinggi')
print('- Kategori shop & entertain paling sering impulsif')